# XGBoost Forex Classifier - Volatility-Weighted MLflow Experiment

Weighted variant of `xgboost_experiment.ipynb`. Samples from high raw-price volatility periods receive lower XGBoost sample weights, while quieter periods receive higher weights.


In [1]:
import os
import json
import tempfile
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import xgboost as xgb
import mlflow
import mlflow.xgboost

from sklearn.metrics import (
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
    log_loss, brier_score_loss, balanced_accuracy_score,
    matthews_corrcoef,
)
from sklearn.calibration import calibration_curve

from Pipeline.pipeline import ForexDataLoader, ForexPipeline, resample_ohlcv, TIMEFRAMES

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## Configuration

Edit `DATA_CFG` and `XGB_PARAMS` here — everything downstream reads from these dicts.

In [ ]:
DATA_CFG = {
    "pair":             "EURUSD",
    "years":            [2022, 2023],
    "timeframe":        "H1",
    "weekends":         "nogap",  # "filled" | "nogap" | "gaps"
    # normalization: "log_returns" | "fracdiff" | "raw"
    "norm_method":      "fracdiff",
    # target: "lag" (direction_1/5/15) | "triple_barrier" (tb_label)
    "target_type":      "triple_barrier",
    # target column: "tb_label" for triple_barrier | "direction_1/5/15" for lag
    "target_col":       "tb_label",
    "lags":             [1, 2, 5, 10],
    "target_horizons":  [1, 5, 15],
    "gap_bars":         50,
    # scaling: "global" (ForexScaler) | "rolling" (RollingScaler),none for no scaling
    "scaling":          "none",
    "window_size":       500,  # only used if scaling="rolling",200-500 is ok
    "fracdiff_d":       0.3,
    "threshold":        6*1e-4,
    "k_up":             2.0,
    "k_down":           1.0,
    "horizon_bars":     10,
    "barrier_price":    "hl",
    "barrier_norm_method": "log_returns",  # "log_returns" | "fracdiff" | "raw"
}

WEIGHT_CFG = {
    "enabled":            True,
    "source":             "raw_rolling",
    "lookback_bars":      100,
    "min_periods":        50,
    "vol_floor_quantile": 0.05,
    "vol_cap_quantile":   0.95,
    "power":              1.0,
    "min_weight":         0.25,
    "max_weight":         2.0,
    "normalize":          "train_mean",
}

OVERLAP_WEIGHT_CFG = {
    "enabled":       True,
    "method":        "lookback_proxy",
    "event_scope":   "class_specific",
    "lookback_bars": DATA_CFG["horizon_bars"],# within how many bars back we looking for neighbour events
    "min_weight":    0.10,
    "max_weight":    1.0,
}

XGB_PARAMS = {
    "num_class":            3,
    "n_estimators":         300,
    "max_depth":            5,
    "learning_rate":        0.05,
    "subsample":            0.8,
    "colsample_bytree":     0.8,
    "min_child_weight":     50,
    "gamma":                0.1,
    "reg_alpha":            0.1,
    "reg_lambda":           1.0,
    "early_stopping_rounds":20,
    "objective":            "multi:softprob",#"binary:logistic",-target lag
    "eval_metric":          "mlogloss",#"logloss"-single
    "random_state":         42,
    "n_jobs":               -1,
}

# MLflow tracking DB - stored inside the project folder
MLFLOW_DB        = f"sqlite:///{os.path.abspath('mlflow.db')}"
EXPERIMENT_NAME  = "xgboost_forex_weighted"

print(f"DATA_CFG  : {json.dumps(DATA_CFG, indent=2)}")
print(f"WEIGHT_CFG: {json.dumps(WEIGHT_CFG, indent=2)}")
print(f"OVERLAP_WEIGHT_CFG: {json.dumps(OVERLAP_WEIGHT_CFG, indent=2)}")
print(f"XGB_PARAMS: {json.dumps(XGB_PARAMS, indent=2)}")
print(f"MLflow DB : {MLFLOW_DB}")


## 1 — Load Raw M1 Data

In [ ]:
loader = ForexDataLoader()
df_m1  = loader.load_and_merge(
    "histdata/",
    pair=DATA_CFG["pair"],
    years=DATA_CFG["years"],weekends=DATA_CFG["weekends"]
)

print(f"Raw M1 shape : {df_m1.shape}")
print(f"Date range   : {df_m1.index.min()}  →  {df_m1.index.max()}")
df_m1.head(3)

## 2 — Run Pipeline

In [ ]:
pipeline = ForexPipeline(
    lags            = DATA_CFG["lags"],
    target_horizons = DATA_CFG["target_horizons"],
    gap_bars        = DATA_CFG["gap_bars"],
    scaling         = DATA_CFG["scaling"],
    norm_method     = DATA_CFG["norm_method"],
    fracdiff_d      = DATA_CFG["fracdiff_d"],
    target_type     = DATA_CFG["target_type"],
    k_up            = DATA_CFG["k_up"],
    k_down          = DATA_CFG["k_down"],
    horizon_bars    = DATA_CFG["horizon_bars"],
    barrier_price   = DATA_CFG.get("barrier_price", "hl"),
    barrier_norm_method = DATA_CFG.get("barrier_norm_method", "raw"),
    threshold       = DATA_CFG["threshold"],
    weekends        = DATA_CFG["weekends"],
)

results     = pipeline.run(df_m1, timeframe=DATA_CFG["timeframe"])
feature_cols = results["feature_cols"]
target_col  = DATA_CFG["target_col"]

for split in ["train", "val", "test"]:
    df_s = results[split]
    print(f"{split:5s}: {len(df_s):6d} rows  "
          f"{df_s.index.min().date()} → {df_s.index.max().date()}")

print(f"\nFeatures ({len(feature_cols)}): {feature_cols}")

## 3 — Extract X / y Arrays

In [ ]:
# Class mapping after triple_barrier +1 remap:
#   0 = short hit   (tb_label = -1 : price hit the lower barrier)
#   1 = neutral     (tb_label =  0 : time expired, no barrier hit)
#   2 = long hit    (tb_label = +1 : price hit the upper barrier)
CLASS_LABELS = {0: "short", 1: "neutral", 2: "long"}

def extract_xy(results, split, target_col, feature_cols):
    X, y = pipeline.get_xy(results[split], target_col, feature_cols)
    if target_col == "tb_label":
        y = (y + 1).astype(int)  # -1→0, 0→1, 1→2
    return X, y.astype(int)

X_train, y_train = extract_xy(results, "train", target_col, feature_cols)
X_val,   y_val   = extract_xy(results, "val",   target_col, feature_cols)
X_test,  y_test  = extract_xy(results, "test",  target_col, feature_cols)

print("Class mapping (triple_barrier):")
for cls, lbl in CLASS_LABELS.items():
    print(f"  {cls} = {lbl}")
print()

for name, y in [("train", y_train), ("val", y_val), ("test", y_test)]:
    total = len(y)
    unique, counts = np.unique(y, return_counts=True)
    dist_str = "  ".join(
        f"{CLASS_LABELS[int(k)]}({int(k)}): {int(v):,} ({int(v)/total:.1%})"
        for k, v in zip(unique, counts)
    )
    print(f"{name:5s}: {total:,} samples  |  {dist_str}")

## 3b - Raw Volatility Sample Weights


In [ ]:
def build_raw_volatility(df_m1, timeframe, lookback_bars, min_periods):
    if timeframe == "M1":
        df_tf = df_m1.copy()
    else:
        df_tf = resample_ohlcv(df_m1, TIMEFRAMES[timeframe])

    close = df_tf["close"].astype(float)
    log_ret = np.log(close).diff()
    return log_ret.rolling(lookback_bars, min_periods=min_periods).std()


def summarize_series_by_split(series_by_split):
    summary = {}
    for name, s in series_by_split.items():
        clean = pd.Series(s).replace([np.inf, -np.inf], np.nan).dropna()
        if clean.empty:
            summary[name] = {"min": None, "max": None, "mean": None, "std": None}
        else:
            summary[name] = {
                "min": float(clean.min()),
                "max": float(clean.max()),
                "mean": float(clean.mean()),
                "std": float(clean.std(ddof=0)),
            }
    return summary


def build_inverse_vol_weights(results, df_m1, weight_cfg):
    split_index = {name: results[name].index for name in ["train", "val", "test"]}
    if not weight_cfg.get("enabled", True):
        weights = {name: pd.Series(1.0, index=idx, name="vol_sample_weight") for name, idx in split_index.items()}
        vol_by_split = {name: pd.Series(np.nan, index=idx, name="raw_volatility") for name, idx in split_index.items()}
        summary = summarize_series_by_split(weights)
        return weights, vol_by_split, summary, {"vol_floor": None, "vol_cap": None, "train_median_vol": None}

    if weight_cfg["source"] != "raw_rolling":
        raise ValueError(f"Unsupported weight source: {weight_cfg['source']}")

    raw_vol = build_raw_volatility(
        df_m1=df_m1,
        timeframe=DATA_CFG["timeframe"],
        lookback_bars=weight_cfg["lookback_bars"],
        min_periods=weight_cfg["min_periods"],
    )

    vol_by_split = {
        name: raw_vol.reindex(idx).ffill().bfill().rename("raw_volatility")
        for name, idx in split_index.items()
    }

    train_vol = vol_by_split["train"].replace([np.inf, -np.inf], np.nan).dropna()
    if train_vol.empty:
        raise ValueError("No finite raw volatility values are available for train weights.")

    vol_floor = float(train_vol.quantile(weight_cfg["vol_floor_quantile"]))
    vol_cap = float(train_vol.quantile(weight_cfg["vol_cap_quantile"]))
    if not np.isfinite(vol_floor) or vol_floor <= 0:
        vol_floor = float(train_vol[train_vol > 0].min())
    if not np.isfinite(vol_cap) or vol_cap <= vol_floor:
        vol_cap = float(train_vol.max())

    train_median_vol = float(train_vol.clip(lower=vol_floor, upper=vol_cap).median())
    if not np.isfinite(train_median_vol) or train_median_vol <= 0:
        train_median_vol = vol_floor

    weights = {}
    for name, vol in vol_by_split.items():
        clean_vol = vol.replace([np.inf, -np.inf], np.nan).fillna(train_median_vol)
        clipped_vol = clean_vol.clip(lower=vol_floor, upper=vol_cap)
        w = (train_median_vol / clipped_vol).pow(weight_cfg["power"])
        w = w.clip(lower=weight_cfg["min_weight"], upper=weight_cfg["max_weight"])
        weights[name] = w.astype(float).rename("vol_sample_weight")

    if weight_cfg.get("normalize") == "train_mean":
        train_mean = float(weights["train"].mean())
        if np.isfinite(train_mean) and train_mean > 0:
            weights = {name: (w / train_mean).rename("vol_sample_weight") for name, w in weights.items()}

    summary = summarize_series_by_split(weights)
    params = {
        "vol_floor": vol_floor,
        "vol_cap": vol_cap,
        "train_median_vol": train_median_vol,
    }
    return weights, vol_by_split, summary, params


def build_overlap_weights(y_by_split, index_by_split, overlap_cfg):
    if overlap_cfg.get("method") != "lookback_proxy":
        raise ValueError(f"Unsupported overlap method: {overlap_cfg['method']}")
    if overlap_cfg.get("event_scope") != "class_specific":
        raise ValueError(f"Unsupported overlap event_scope: {overlap_cfg['event_scope']}")

    lookback_bars = int(overlap_cfg["lookback_bars"])
    if lookback_bars < 0:
        raise ValueError("overlap lookback_bars must be >= 0")

    weights = {}
    concurrency = {}
    for name, y in y_by_split.items():
        y_s = pd.Series(y, index=index_by_split[name])
        if not overlap_cfg.get("enabled", True):
            weights[name] = pd.Series(1.0, index=y_s.index, name="overlap_sample_weight")
            concurrency[name] = pd.Series(0.0, index=y_s.index, name="overlap_concurrency")
            continue

        short_counts = (y_s == 0).astype(float).rolling(lookback_bars + 1, min_periods=1).sum()
        long_counts = (y_s == 2).astype(float).rolling(lookback_bars + 1, min_periods=1).sum()
        c = pd.Series(0.0, index=y_s.index, name="overlap_concurrency")
        c.loc[y_s == 0] = short_counts.loc[y_s == 0]
        c.loc[y_s == 2] = long_counts.loc[y_s == 2]

        w = pd.Series(1.0, index=y_s.index, name="overlap_sample_weight")
        directional = c > 0
        w.loc[directional] = 1.0 / c.loc[directional]
        w = w.clip(lower=overlap_cfg["min_weight"], upper=overlap_cfg["max_weight"])

        weights[name] = w.astype(float).rename("overlap_sample_weight")
        concurrency[name] = c.astype(float).rename("overlap_concurrency")

    weight_summary = summarize_series_by_split(weights)
    concurrency_summary = summarize_series_by_split(concurrency)
    return weights, concurrency, weight_summary, concurrency_summary


def combine_sample_weights(vol_weights, overlap_weights, normalize="train_mean"):
    combined = {
        name: (vol_weights[name] * overlap_weights[name]).rename("sample_weight")
        for name in vol_weights
    }
    if normalize == "train_mean":
        train_mean = float(combined["train"].mean())
        if np.isfinite(train_mean) and train_mean > 0:
            combined = {name: (w / train_mean).rename("sample_weight") for name, w in combined.items()}
    return combined, summarize_series_by_split(combined)


vol_sample_weights, raw_volatility, vol_weight_summary, weight_params = build_inverse_vol_weights(
    results=results,
    df_m1=df_m1,
    weight_cfg=WEIGHT_CFG,
)

overlap_sample_weights, overlap_concurrency, overlap_weight_summary, overlap_concurrency_summary = build_overlap_weights(
    y_by_split={"train": y_train, "val": y_val, "test": y_test},
    index_by_split={name: results[name].index for name in ["train", "val", "test"]},
    overlap_cfg=OVERLAP_WEIGHT_CFG,
)

sample_weights, weight_summary = combine_sample_weights(
    vol_weights=vol_sample_weights,
    overlap_weights=overlap_sample_weights,
    normalize="train_mean",
)

w_train = sample_weights["train"].to_numpy()
w_val   = sample_weights["val"].to_numpy()
w_test  = sample_weights["test"].to_numpy()

for name, X, w in [("train", X_train, w_train), ("val", X_val, w_val), ("test", X_test, w_test)]:
    if len(w) != len(X):
        raise ValueError(f"{name} weights length mismatch: weights={len(w)} X={len(X)}")
    if not np.isfinite(w).all() or (w <= 0).any():
        raise ValueError(f"{name} weights must be finite and positive")

print("Raw-volatility inverse sample weights:")
print(pd.DataFrame(vol_weight_summary).T.round(4))
print("\nClass-specific overlap sample weights:")
print(pd.DataFrame(overlap_weight_summary).T.round(4))
print("\nFinal multiplied sample weights:")
print(pd.DataFrame(weight_summary).T.round(4))
print("\nOverlap concurrency:")
print(pd.DataFrame(overlap_concurrency_summary).T.round(4))
print("\nVolatility clipping params:")
print(pd.Series(weight_params).to_string())

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for name, color in [("train", "steelblue"), ("val", "darkorange"), ("test", "seagreen")]:
    sns.histplot(sample_weights[name], bins=40, kde=True, stat="density", ax=axes[0], label=name, color=color, alpha=0.25)
axes[0].set_title("Final Sample Weight Distribution")
axes[0].set_xlabel("sample_weight")
axes[0].legend()

for name, color in [("train", "steelblue"), ("val", "darkorange"), ("test", "seagreen")]:
    aligned = pd.DataFrame({
        "raw_volatility": raw_volatility[name],
        "vol_sample_weight": vol_sample_weights[name],
    })
    axes[1].scatter(aligned["raw_volatility"], aligned["vol_sample_weight"], s=8, alpha=0.25, label=name, color=color)
axes[1].set_title("Raw Volatility vs Vol Weight")
axes[1].set_xlabel("rolling raw log-return volatility")
axes[1].set_ylabel("vol_sample_weight")
axes[1].legend()

for name, color in [("train", "steelblue"), ("val", "darkorange"), ("test", "seagreen")]:
    aligned = pd.DataFrame({
        "overlap_concurrency": overlap_concurrency[name],
        "overlap_sample_weight": overlap_sample_weights[name],
    })
    axes[2].scatter(aligned["overlap_concurrency"], aligned["overlap_sample_weight"], s=8, alpha=0.25, label=name, color=color)
axes[2].set_title("Overlap Concurrency vs Overlap Weight")
axes[2].set_xlabel("same-class trailing concurrency")
axes[2].set_ylabel("overlap_sample_weight")
axes[2].legend()
fig.tight_layout()


## 4 — Train XGBoost + Log Everything to MLflow

In [ ]:
mlflow.set_tracking_uri(MLFLOW_DB)
mlflow.set_experiment(EXPERIMENT_NAME)

run_name = f"{DATA_CFG['pair']}_{DATA_CFG['timeframe']}_{DATA_CFG['norm_method']}_{DATA_CFG['target_col']}_rawvol_weighted"

with mlflow.start_run(run_name=run_name) as run:
    RUN_ID = run.info.run_id

    # ── Parameters ──────────────────────────────────────────────────────────
    mlflow.log_params({f"data_{k}": str(v) for k, v in DATA_CFG.items()})
    mlflow.log_params({f"xgb_{k}": v for k, v in XGB_PARAMS.items()})
    mlflow.log_params({f"weight_{k}": str(v) for k, v in WEIGHT_CFG.items()})
    mlflow.log_params({f"overlap_weight_{k}": str(v) for k, v in OVERLAP_WEIGHT_CFG.items()})

    # Data date ranges
    mlflow.log_params({
        "raw_m1_start":  str(df_m1.index.min()),
        "raw_m1_end":    str(df_m1.index.max()),
        "train_start":   str(results["train"].index.min()),
        "train_end":     str(results["train"].index.max()),
        "val_start":     str(results["val"].index.min()),
        "val_end":       str(results["val"].index.max()),
        "test_start":    str(results["test"].index.min()),
        "test_end":      str(results["test"].index.max()),
        "n_features":    len(feature_cols),
        "train_samples": len(X_train),
        "val_samples":   len(X_val),
        "test_samples":  len(X_test),
        "train_pos_rate":float((y_train == 2).mean()),
        "val_pos_rate":  float((y_val   == 2).mean()),
        "test_pos_rate": float((y_test  == 2).mean()),
        "weight_vol_floor": weight_params["vol_floor"],
        "weight_vol_cap": weight_params["vol_cap"],
        "weight_train_median_vol": weight_params["train_median_vol"],
    })

    # Feature list and weight diagnostics as artifacts
    mlflow.log_dict({"features": feature_cols}, "features.json")
    mlflow.log_dict({
        "volatility_config": WEIGHT_CFG,
        "overlap_config": OVERLAP_WEIGHT_CFG,
        "volatility_summary": vol_weight_summary,
        "overlap_summary": overlap_weight_summary,
        "overlap_concurrency_summary": overlap_concurrency_summary,
        "final_summary": weight_summary,
        "params": weight_params,
    }, "weights.json")
    mlflow.log_metrics({
        f"{split}_weight_{metric}": value
        for split, stats in weight_summary.items()
        for metric, value in stats.items()
        if value is not None
    })
    mlflow.log_metrics({
        f"{split}_vol_weight_{metric}": value
        for split, stats in vol_weight_summary.items()
        for metric, value in stats.items()
        if value is not None
    })
    mlflow.log_metrics({
        f"{split}_overlap_weight_{metric}": value
        for split, stats in overlap_weight_summary.items()
        for metric, value in stats.items()
        if value is not None
    })
    mlflow.log_metrics({
        f"{split}_overlap_concurrency_{metric}": value
        for split, stats in overlap_concurrency_summary.items()
        for metric, value in stats.items()
        if value is not None
    })

    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "weight_distribution.png")
        fig.savefig(path, bbox_inches="tight")
        mlflow.log_artifact(path, artifact_path="plots")

    # ── Train ────────────────────────────────────────────────────────────────
    model = xgb.XGBClassifier(**XGB_PARAMS)
    model.fit(
        X_train, y_train,
        sample_weight=w_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        sample_weight_eval_set=[w_train, w_val],
        verbose=False,
    )

    mlflow.log_param("best_iteration", model.best_iteration)

    # ── Log model ────────────────────────────────────────────────────────────
    mlflow.xgboost.log_model(model, "model")

    # ── Metrics per split ────────────────────────────────────────────────────
    # proba shape: (n, 3)  →  [P(short/0), P(neutral/1), P(long/2)]
    split_metrics = {}
    for sname, X, y in [("train", X_train, y_train),
                         ("val",   X_val,   y_val),
                         ("test",  X_test,  y_test)]:
        proba      = model.predict_proba(X)          # (n, 3)
        proba_long = proba[:, 2]                     # P(long hit) for binary metrics
        y_long     = (y == 2).astype(int)
        pred       = model.predict(X)
        m = {
            "auc":           roc_auc_score(y, proba, multi_class="ovr", average="macro"),
            "auc_long":      roc_auc_score(y_long, proba_long),
            "avg_precision": average_precision_score(y, proba, average="macro"),
            "logloss":       log_loss(y, proba),
            "brier_long":    brier_score_loss(y_long, proba_long),
            "f1":            f1_score(y, pred, average="macro", zero_division=0),
            "precision":     precision_score(y, pred, average="macro", zero_division=0),
            "recall":        recall_score(y, pred, average="macro", zero_division=0),
            "balanced_acc":  balanced_accuracy_score(y, pred),
            "mcc":           matthews_corrcoef(y, pred),
        }
        split_metrics[sname] = m
        mlflow.log_metrics({f"{sname}_{k}": v for k, v in m.items()})

    print(f"Run ID  : {RUN_ID}")
    print(f"Best iter: {model.best_iteration}")
    print(pd.DataFrame(split_metrics).T.round(4))

## 5 — Analysis: ROC Curves (train / val / test)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

colors = {"train": "steelblue", "val": "darkorange", "test": "seagreen"}
styles = {"train": "--",        "val": "-",          "test": "-"}

# Plot ROC for "long hit" class (class 2) — most actionable signal
for sname, X, y in [("train", X_train, y_train),
                     ("val",   X_val,   y_val),
                     ("test",  X_test,  y_test)]:
    proba_long = model.predict_proba(X)[:, 2]
    y_long     = (y == 2).astype(int)
    fpr, tpr, _ = roc_curve(y_long, proba_long)
    auc = roc_auc_score(y_long, proba_long)
    ax.plot(fpr, tpr, ls=styles[sname], color=colors[sname],
            label=f"{sname}  AUC = {auc:.4f}", lw=2)

ax.plot([0, 1], [0, 1], 'k:', lw=1, label="random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title(f"ROC Curve (long hit) — {DATA_CFG['pair']} {DATA_CFG['timeframe']} ({DATA_CFG['target_col']})")
ax.legend(loc="lower right")
plt.tight_layout()

with mlflow.start_run(run_id=RUN_ID):
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "roc_curve.png")
        fig.savefig(path)
        mlflow.log_artifact(path, "plots")

plt.show()

## 6 — Analysis: Precision-Recall Curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

# Plot PR for "long hit" class (class 2)
for sname, X, y in [("val",  X_val,  y_val),
                     ("test", X_test, y_test)]:
    proba_long = model.predict_proba(X)[:, 2]
    y_long     = (y == 2).astype(int)
    prec, rec, _ = precision_recall_curve(y_long, proba_long)
    ap = average_precision_score(y_long, proba_long)
    ax.plot(rec, prec, color=colors[sname],
            label=f"{sname}  AP = {ap:.4f}", lw=2)

# No-skill baseline (long-hit base rate)
for sname, y in [("val", y_val), ("test", y_test)]:
    ax.axhline((y == 2).mean(), ls=':', color=colors[sname], alpha=0.5)

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title(f"Precision-Recall (long hit) — {DATA_CFG['pair']} {DATA_CFG['timeframe']}")
ax.legend()
plt.tight_layout()

with mlflow.start_run(run_id=RUN_ID):
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "pr_curve.png")
        fig.savefig(path)
        mlflow.log_artifact(path, "plots")

plt.show()

## 7 — Analysis: Confusion Matrices (val + test)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm_labels = [f"{CLASS_LABELS[i]} ({i})" for i in range(3)]

for ax, (sname, X, y) in zip(axes, [("val", X_val, y_val), ("test", X_test, y_test)]):
    pred    = model.predict(X)
    cm      = confusion_matrix(y, pred, labels=[0, 1, 2])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    annot   = np.array([[f"{cm[i,j]}\n({cm_norm[i,j]:.1%})" for j in range(3)] for i in range(3)])
    sns.heatmap(cm_norm, annot=annot, fmt='', cmap='Blues', ax=ax,
                xticklabels=[f"pred {l}" for l in cm_labels],
                yticklabels=[f"true {l}" for l in cm_labels],
                vmin=0, vmax=1, linewidths=0.5)
    ax.set_title(f"Confusion Matrix — {sname}")

plt.tight_layout()

with mlflow.start_run(run_id=RUN_ID):
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "confusion_matrices.png")
        fig.savefig(path)
        mlflow.log_artifact(path, "plots")

plt.show()

## 8 — Analysis: Classification Report

In [ ]:
report_frames = []
_target_names = [CLASS_LABELS[i] for i in range(3)]   # ["short", "neutral", "long"]

for sname, X, y in [("val", X_val, y_val), ("test", X_test, y_test)]:
    pred = model.predict(X)

    # Class distribution
    total = len(y)
    unique, counts = np.unique(y, return_counts=True)
    dist = {CLASS_LABELS[int(k)]: int(v) for k, v in zip(unique, counts)}
    print(f"\n── {sname}  (n={total:,}) ──")
    print("  Class counts: " + "  ".join(f"{lbl}: {cnt:,} ({cnt/total:.1%})"
                                          for lbl, cnt in dist.items()))

    print(classification_report(y, pred, target_names=_target_names))

    rpt    = classification_report(y, pred, target_names=_target_names, output_dict=True)
    df_rpt = pd.DataFrame(rpt).T.round(4)
    df_rpt.insert(0, "split", sname)
    report_frames.append(df_rpt)

combined_report = pd.concat(report_frames)

with mlflow.start_run(run_id=RUN_ID):
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "classification_report.csv")
        combined_report.to_csv(path)
        mlflow.log_artifact(path, "reports")

## 9b — Per-class Prediction Quality

In [ ]:
from sklearn.preprocessing import label_binarize

_classes = list(CLASS_LABELS.keys())   # [0, 1, 2]
_splits  = [("val", X_val, y_val), ("test", X_test, y_test)]
_clr     = {"val": "steelblue", "test": "darkorange"}

# ── Per-class ROC curves (One-vs-Rest) ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for sname, X, y in _splits:
    proba = model.predict_proba(X)
    y_bin = label_binarize(y, classes=_classes)
    for ci in range(3):
        auc = roc_auc_score(y_bin[:, ci], proba[:, ci])
        fpr, tpr, _ = roc_curve(y_bin[:, ci], proba[:, ci])
        axes[ci].plot(fpr, tpr, color=_clr[sname], lw=2,
                      label=f"{sname}  AUC={auc:.3f}")

for ci, cls in enumerate(_classes):
    axes[ci].plot([0, 1], [0, 1], "k:", lw=1)
    axes[ci].set_title(f"Class {cls}: {CLASS_LABELS[cls]} (OvR)")
    axes[ci].set_xlabel("FPR")
    axes[ci].set_ylabel("TPR")
    axes[ci].legend(fontsize=9)

fig.suptitle("Per-Class ROC Curves (One-vs-Rest)", fontsize=13)
plt.tight_layout()

with mlflow.start_run(run_id=RUN_ID):
    with tempfile.TemporaryDirectory() as tmp:
        _p = os.path.join(tmp, "per_class_roc.png")
        fig.savefig(_p)
        mlflow.log_artifact(_p, "plots")
plt.show()

# ── Per-class Precision / Recall / F1 bar chart ───────────────────────────────
_x      = np.arange(3)
_w      = 0.35
_labels = [f"{CLASS_LABELS[c]}\n({c})" for c in _classes]

fig2, axes2 = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
for ax, metric_fn, title in zip(
    axes2,
    [precision_score, recall_score, f1_score],
    ["Precision", "Recall", "F1"],
):
    for offset, (sname, X, y) in zip([-_w / 2, _w / 2], _splits):
        scores = metric_fn(y, model.predict(X), average=None, zero_division=0)
        ax.bar(_x + offset, scores, _w, label=sname, color=_clr[sname], alpha=0.85)
    ax.set_xticks(_x)
    ax.set_xticklabels(_labels)
    ax.set_ylim(0, 1.05)
    ax.set_title(title)
    ax.legend(fontsize=9)

fig2.suptitle("Per-Class Precision / Recall / F1  (val vs test)", fontsize=13)
plt.tight_layout()

with mlflow.start_run(run_id=RUN_ID):
    with tempfile.TemporaryDirectory() as tmp:
        _p2 = os.path.join(tmp, "per_class_prf1.png")
        fig2.savefig(_p2)
        mlflow.log_artifact(_p2, "plots")
plt.show()

## 9 — Analysis: Learning Curves (Overfitting Diagnostic)

In [ ]:
evals = model.evals_result()
# keys: 'validation_0' = train, 'validation_1' = val
train_logloss = evals["validation_0"]["logloss"]
val_logloss   = evals["validation_1"]["logloss"]
rounds = range(1, len(train_logloss) + 1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(rounds, train_logloss, label="train logloss", color="steelblue", lw=1.5)
ax.plot(rounds, val_logloss,   label="val logloss",   color="darkorange", lw=1.5)
ax.axvline(model.best_iteration, ls='--', color='seagreen', lw=1.5,
           label=f"best iter = {model.best_iteration}")

# Shade overfitting region
if model.best_iteration < len(train_logloss):
    ax.axvspan(model.best_iteration, len(train_logloss), alpha=0.08, color='red',
               label="overfitting region")

ax.set_xlabel("Boosting Round")
ax.set_ylabel("Log Loss")
ax.set_title(f"Learning Curves — {DATA_CFG['pair']} {DATA_CFG['timeframe']}")
ax.legend()
plt.tight_layout()

# Log overfit gap as metric
overfit_gap = val_logloss[model.best_iteration - 1] - train_logloss[model.best_iteration - 1]
with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_metric("overfit_logloss_gap", overfit_gap)
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "learning_curves.png")
        fig.savefig(path)
        mlflow.log_artifact(path, "plots")

print(f"Val-Train logloss gap at best iter: {overfit_gap:.6f}")
plt.show()

## 10 — Analysis: Feature Importance (Gain / Weight / Cover)

In [ ]:
importance_types = ["gain", "weight", "cover"]
imp_frames = {}

for imp_type in importance_types:
    scores = model.get_booster().get_score(importance_type=imp_type)
    df_imp = pd.Series(scores, name=imp_type).reindex(feature_cols).fillna(0)
    df_imp = df_imp.sort_values(ascending=False)
    imp_frames[imp_type] = df_imp

imp_combined = pd.DataFrame(imp_frames)
imp_combined.index.name = "feature"

fig, axes = plt.subplots(1, 3, figsize=(16, 7))
for ax, imp_type in zip(axes, importance_types):
    top20 = imp_frames[imp_type].head(20)
    top20.plot.barh(ax=ax, color="steelblue")
    ax.invert_yaxis()
    ax.set_title(f"Importance: {imp_type}")
    ax.set_xlabel("Score")

plt.suptitle(f"Feature Importance — {DATA_CFG['pair']} {DATA_CFG['timeframe']}", y=1.01)
plt.tight_layout()

with mlflow.start_run(run_id=RUN_ID):
    with tempfile.TemporaryDirectory() as tmp:
        img_path = os.path.join(tmp, "feature_importance.png")
        csv_path = os.path.join(tmp, "feature_importance.csv")
        fig.savefig(img_path)
        imp_combined.to_csv(csv_path)
        mlflow.log_artifact(img_path, "plots")
        mlflow.log_artifact(csv_path, "reports")

plt.show()
imp_combined.head(10)

## 11 — Analysis: Calibration Curve (Reliability Diagram)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], 'k--', lw=1, label="perfect calibration")

brier_scores = {}
for sname, X, y in [("val", X_val, y_val), ("test", X_test, y_test)]:
    proba_long = model.predict_proba(X)[:, 2]
    y_long     = (y == 2).astype(int)
    frac_pos, mean_pred = calibration_curve(y_long, proba_long, n_bins=10, strategy="uniform")
    brier = brier_score_loss(y_long, proba_long)
    brier_scores[sname] = brier
    ax.plot(mean_pred, frac_pos, 's-', color=colors[sname], lw=2,
            label=f"{sname}  Brier = {brier:.4f}")

ax.set_xlabel("Mean Predicted P(long hit)")
ax.set_ylabel("Fraction of Positives")
ax.set_title(f"Calibration Curve (long hit) — {DATA_CFG['pair']} {DATA_CFG['timeframe']}")
ax.legend()
plt.tight_layout()

with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_metrics({f"{k}_brier": v for k, v in brier_scores.items()})
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "calibration_curve.png")
        fig.savefig(path)
        mlflow.log_artifact(path, "plots")

plt.show()

## 12 — Analysis: Metrics Comparison (Train / Val / Test)

In [ ]:
metrics_df = pd.DataFrame(split_metrics).T
metrics_df.index.name = "split"

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
plot_metrics = [
    (["auc", "auc_long", "avg_precision", "balanced_acc"], "Discrimination"),
    (["f1", "precision", "recall"],                        "Classification (macro)"),
    (["logloss", "brier_long"],                            "Calibration (lower = better)"),
]

for ax, (cols, title) in zip(axes, plot_metrics):
    metrics_df[cols].plot.bar(ax=ax, rot=0)
    ax.set_title(title)
    ax.set_ylim(0)
    ax.legend(fontsize=8)

plt.suptitle("Train / Val / Test Metric Comparison", y=1.02)
plt.tight_layout()

with mlflow.start_run(run_id=RUN_ID):
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "metrics_comparison.png")
        fig.savefig(path)
        mlflow.log_artifact(path, "plots")

plt.show()
print(metrics_df.round(4).to_string())

## 13 — Run Summary

In [ ]:
required_summary_vars = ["RUN_ID", "split_metrics", "EXPERIMENT_NAME", "MLFLOW_DB", "DATA_CFG", "WEIGHT_CFG", "OVERLAP_WEIGHT_CFG", "weight_summary", "pd"]
missing_summary_vars = [name for name in required_summary_vars if name not in globals()]

if missing_summary_vars:
    print("Run summary is not available yet.")
    print("Missing variables: " + ", ".join(missing_summary_vars))
    print("Run the training/MLflow cell first, then rerun the analysis cells.")
else:
    print("=" * 60)
    print(f"  MLflow Run ID  : {RUN_ID}")
    print(f"  Experiment     : {EXPERIMENT_NAME}")
    print(f"  Tracking DB    : {MLFLOW_DB}")
    print(f"  Primary run    : {DATA_CFG['norm_method']} (d={DATA_CFG.get('fracdiff_d')})")
    print(f"  Weight source  : {WEIGHT_CFG['source']} | lookback={WEIGHT_CFG['lookback_bars']} bars")
    print(f"  Overlap source : {OVERLAP_WEIGHT_CFG['event_scope']} | lookback={OVERLAP_WEIGHT_CFG['lookback_bars']} bars")
    print("=" * 60)
    print("  To browse the UI:")
    print(f"  mlflow ui --backend-store-uri {MLFLOW_DB}")
    print("  then open  http://127.0.0.1:5000")
    print("=" * 60)
    print()
    print("Logged artifacts:")
    for artifact in [
        "plots/roc_curve.png",
        "plots/pr_curve.png",
        "plots/confusion_matrices.png",
        "plots/learning_curves.png",
        "plots/feature_importance.png",
        "plots/calibration_curve.png",
        "plots/metrics_comparison.png",
        "plots/weight_distribution.png",
        "reports/classification_report.csv",
        "reports/feature_importance.csv",
        "features.json",
        "weights.json",
        "model/  (XGBoost model)",
    ]:
        print(f"  {artifact}")

    print()
    test_metrics = split_metrics.get("test", {})
    if test_metrics:
        print("Test metrics:")
        print(pd.Series(test_metrics).round(4).to_string())
    else:
        print("No test metrics were found in split_metrics.")